# 4. Stage 4. ESCO Skills Extraction

**Stage:** 4 of 5

**Purpose:** Verify and enrich each vacancy's ESCO occupation assignment from Stage 3. The LLM-generated occupation title is matched against the official ESCO taxonomy using a 4-step pipeline (exact preferred label → exact altLabel → fuzzy preferred label → fuzzy altLabel). Once a verified ESCO title is found, the associated skill list is retrieved from the ESCO occupation–skill relations table.

**Input:**
- Stage 3 result pickles: `data/stage_03/processed/result/ua-YYYY-MM-DD.pkl`
- ESCO reference CSVs: `data/stage_04/raw/esco_data/`

**Output:** `data/stage_04/processed/output/ua-YYYY-MM-DD.pkl` — one pickle per day, with all Stage 3 columns plus `esco_title`, `esco_id`, `esco_code`, `esco_skills`, `extract_type`.

**Run:** Execute all cells top to bottom. No API calls required — this stage uses only local data and fuzzy matching.

## 4.1. Load packages and configuration

Enable autoreload so that changes to `code/` modules are picked up automatically without restarting the kernel. Add the `code/` directory to the Python path, then import `general` to trigger memory cleanup via `gc.collect()`.

In [2]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append("../code")
import general as g
g.clean_memory()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Import pipeline modules and libraries:
- `stage4` — ESCO lookup helpers (`find_closest_occupation_title`, `find_closest_occupation_code`) and `manual_data_correction()`
- `rapidfuzz` — fast fuzzy string matching used in the 4-step ESCO title verification pipeline
- `time` — used to log execution time per daily file

In [4]:
import stage4 as st4
import pandas as pd
import json
import time
from rapidfuzz import fuzz, process

## 4.2. Load ESCO occupation data

Load the three ESCO reference tables from `data/stage_04/raw/esco_data/` (ESCO v1.2.0, English):
- `occupations_en.csv` — 3 000+ occupation concepts with preferred labels, alternative labels, 4-digit codes, and concept URIs
- `skills_en.csv` — 13 800+ skill/knowledge concepts
- `occupationSkillRelations_en.csv` — links each occupation URI to its essential and optional skill URIs

These files are static reference data and do not change between pipeline runs.

In [5]:
occupation_skills_df = pd.read_csv(os.path.join(g.Config.STAGE4_ESCO_DATA_PATH, 'occupationSkillRelations_en.csv'))
occupations_df = pd.read_csv(os.path.join(g.Config.STAGE4_ESCO_DATA_PATH, 'occupations_en.csv'))
skills_df = pd.read_csv(os.path.join(g.Config.STAGE4_ESCO_DATA_PATH, 'skills_en.csv'))

Preview the occupation and skill tables to confirm the expected column structure (`preferredLabel`, `altLabels`, `code`, `conceptUri`).

In [6]:
occupations_df.head(2)

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,technical and operations director\nhead of tec...,NaN,released,2024-01-25T11:28:50.295Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Technical directors realise the artistic visio...,2654.1.7
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,metal drawing machine technician\nmetal drawin...,NaN,released,2024-01-23T10:09:32.099Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4


In [7]:
skills_df.head(2)

,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage staff of music\ncoordinate duties of mu...,NaN,released,2023-11-30T15:53:37.136Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,oversee prison procedures\nmanage correctional...,NaN,released,2023-11-30T15:04:00.689Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Supervise the operations of a correctional fac...


## 4.3. Load previous stage logs

Read the Stage 3 process tracker (`data/stage_03/intermediate/process.pkl`), sort by filename, and reset the index. Only rows where `result_status == 'complete'` will be processed — those are daily files that passed both Batch API passes in Stage 3.

In [8]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
process_df = process_df.sort_values(by='input_file')
process_df = process_df.reset_index(drop=True)
process_df.head()

,input_file,extract_path,input_batch_path,input_batch_status,job_id,job_status,output_batch_path,output_batch_status,result_path,result_status
0,ua-2024-01-01,../data/stage_02/processed/output\ua-2024-01-0...,../data/stage_03/raw/input\ua-2024-01-01.jsonl,created,batch_69c055917b0481909b82ddb32c228250,completed,../data/stage_03/processed/output\ua-2024-01-0...,complete,../data/stage_03/processed/result\ua-2024-01-0...,complete


Ensure the Stage 4 output folder (`data/stage_04/processed/output/`) exists, creating it if needed.

In [9]:
g.check_folder_exists(g.Config.STAGE4_OUTPUT_PATH)

Helper function `extra_clean_text()` normalises occupation title strings: strip surrounding whitespace, convert to lowercase, collapse internal whitespace. Used to make LLM-generated titles comparable to ESCO labels during exact and fuzzy matching.

In [10]:
def extra_clean_text(x):
    return ' '.join(x.strip().lower().split())

Build the `esco_exploded` lookup index. ESCO occupations have a `altLabels` column containing newline-separated synonyms (e.g. `"software engineer\nprogrammer\ndeveloper"`). This cell explodes that column so each alternative label occupies its own row. Labels are normalised to lowercase. The resulting DataFrame is the index for Step 2 and Step 4 of the matching pipeline (altLabel exact and fuzzy matching).

In [11]:
esco_exploded = occupations_df.copy()
esco_exploded['altLabels'] = esco_exploded['altLabels'].fillna('').str.split('\n')
esco_exploded = esco_exploded.explode('altLabels')
esco_exploded['altLabels'] = esco_exploded['altLabels'].str.strip().str.lower()
esco_exploded = esco_exploded[['preferredLabel', 'altLabels']]
esco_exploded.drop_duplicates(subset = ['altLabels'], keep = 'first', inplace=True)
esco_exploded

,preferredLabel,altLabels
0,technical director,technical and operations director
0,technical director,head of technical
0,technical director,director of technical arts
0,technical director,head of technical department
0,technical director,technical supervisor
...,...,...
3038,motor vehicle assembler,motor vehicle assemblies tester
3038,motor vehicle assembler,car factory worker
3038,motor vehicle assembler,motor vehicle assembly inspector
3038,motor vehicle assembler,truck production line worker


Inspect the alternative labels list to verify the exploded index is built correctly. Expected output: a flat list of tens of thousands of lowercase strings.

In [12]:
esco_exploded["altLabels"].tolist()

['technical and operations director',
 'head of technical',
 'director of technical arts',
 'head of technical department',
 'technical supervisor',
 'technical manager',
 'metal drawing machine technician',
 'metal drawing machine operative',
 'wire drawer',
 'draw machine operative',
 'forming machine technician',
 'draw machine operator',
 'wiredrawing setter',
 'wirer drawer machine operator',
 'forming machine operative',
 'draw machine technician',
 'wiredrawing  machine tender',
 'inspector of precision instruments',
 'precision device quality control supervisor',
 'precision instrument qc inspector',
 'precision instrument quality control inspector',
 'precision device qc inspector',
 'precision device quality assurance supervisor',
 'precision device quality control inspector',
 'inspector of precision devices',
 'precision instrument inspector',
 'precision instrument supervisor',
 'air traffic safety electronics hardware specialist',
 'air traffic safety software specialist'

Build flat Python lists of all unique preferred labels and alternative labels. These are passed directly to `rapidfuzz.process.extractOne()` in Steps 3 and 4 for fast vocabulary-wide fuzzy search.

In [13]:
esco_labels = esco_exploded['preferredLabel'].dropna().unique().tolist()
alt_labels = esco_exploded['altLabels'].dropna().unique().tolist()
esco_labels

['technical director',
 'metal drawing machine operator',
 'precision device inspector',
 'air traffic safety technician',
 'hospitality revenue manager',
 'medical laboratory assistant',
 'asphalt laboratory technician',
 'primary school teaching assistant',
 'physiotherapist',
 'performing arts theatre instructor',
 'hydrogenation machine operator',
 'pasta operator',
 'tobacco shop manager',
 'rental service representative in other machinery, equipment and tangible goods',
 'mine surveyor',
 'food safety specialist',
 'land-based machinery technician',
 'bus driver',
 'import export manager in meat and meat products',
 'housing policy officer',
 'head chef',
 'nonwoven filament machine operator',
 'head of higher education institutions',
 'insulation supervisor',
 'dismantling engineer',
 'insurance claims handler',
 'clothing development manager',
 'medical writer',
 'nitroglycerin separator operator',
 'biomedical engineer',
 'legal guardian',
 'weaver',
 'tile fitter',
 'shoe and

Five helper functions used inside the main processing loop:
- `get_preferred_label(alt_label_value, df)` — resolves an alternative label back to its parent preferred label
- `get_esco_id(title, df)` — returns the full ESCO concept URI for a given preferred label
- `get_esco_code(title, df)` — returns the 4-digit ESCO occupation code for a given preferred label
- `get_best_match(label, choices, threshold=80)` — fuzzy match against the preferred-label list using `token_sort_ratio`; returns `None` if similarity < 80
- `get_best_alt_match(label, choices_df, threshold=80)` — fuzzy match against all alternative labels, then resolves to the preferred label

In [14]:
def get_preferred_label(alt_label_value, df):
    for _, row in df.iterrows():
        if alt_label_value in row['altLabels']:
            return row['preferredLabel']
    return None

def get_esco_id(title, df):
    for _, row in df.iterrows():
        if title in row['preferredLabel']:
            return row['conceptUri']
    return None

def get_esco_code(title, df):
    for _, row in df.iterrows():
        if title in row['preferredLabel']:
            return row['code']
    return None

def get_best_match(label, choices, threshold=80):
    result = process.extractOne(label, choices, scorer=fuzz.token_sort_ratio)
    if result and result[1] >= threshold:
        return result[0]
    return None

def get_best_alt_match(label, choices_df, threshold=80):
    result = process.extractOne(label, choices_df["altLabels"].tolist(), scorer=fuzz.token_sort_ratio)
    if result and result[1] >= threshold:
        return get_preferred_label(result[0], choices_df)
    return None

## 4.4. Extract skills and add to main daily database

**Main processing loop** — iterates over each Stage 3 completed file and applies a **4-step ESCO title verification and skill extraction** pipeline. Files that already have an output pickle in `data/stage_04/processed/output/` are skipped automatically.

**Matching pipeline (applied to each vacancy in turn):**

| Step | Method | Description |
|------|--------|-------------|
| 1 | `preferredLabel` exact | Cleaned LLM title matched directly against ESCO preferred labels |
| 2 | `altLabels` exact | Unresolved records checked against the exploded altLabel index |
| 3 | `preferredLabel` fuzzy | `rapidfuzz.token_sort_ratio` (threshold 80) against all preferred labels |
| 4 | `altLabels` fuzzy | Fuzzy match against all alternative labels, resolved to preferred label |

**Columns added to each record:**
- `esco_title` — verified ESCO preferred label
- `esco_id` — full ESCO concept URI
- `esco_code` — 4-digit ESCO occupation code
- `esco_skills` — JSON list of all skills linked to this occupation
- `extract_type` — which step resolved the match

Records that could not be matched in any step are retained with `None` in the enrichment columns.

> **Demo note:** The loop slice `process_df[0:3]` processes up to the first 3 files. For the single demo file, only index 0 will be present.

In [15]:
stage3_data = None
merge_step1 = None

for pindex, prow in process_df.iterrows():

    if prow['result_status'] == 'complete':
        start_time = time.time()
        output_path = os.path.join(Config.STAGE4_OUTPUT_PATH, prow["input_file"] + ".pkl")

        if os.path.exists(output_path):
            print(f"{pindex} ----- File skipped for {prow['input_file']}")
            continue

        stage3_data = pd.read_pickle(process_df.loc[pindex, 'result_path']).reset_index(drop=True)

        stage3_data = stage3_data.rename(columns={'esco_code': 'classified_code', 'esco_title': 'classified_title'})

        print(f"{pindex} ----- Working on {prow['input_file']} - {stage3_data.shape[0]}")
        stage3_data['skill_labels_en'] = None

        stage3_data["classified_title_clean"] = stage3_data["classified_title"].apply(extra_clean_text)
        stage3_data["extract_type"] = None

        # Step 1: Extract match on preferredLabel
        merge_step1 = stage3_data[['id', 'classified_title_clean']].merge(occupations_df[["preferredLabel"]], left_on='classified_title_clean', right_on='preferredLabel', how='left')
        stage3_data = stage3_data.merge(merge_step1[['id', 'preferredLabel']], left_on='id', right_on='id', how='left')
        stage3_data.rename({'preferredLabel': 'esco_title'}, axis=1, inplace=True)

        ready_data = stage3_data[stage3_data['esco_title'].notna()].copy()
        ready_data['extract_type'] = "preferredLabel"
        not_ready_data = stage3_data[stage3_data['esco_title'].isna()].copy()
        not_ready_data.drop(columns=['esco_title'], inplace=True)

        #print(f"ready: {ready_data.shape[0]} - not ready {not_ready_data.shape[0]} - {merge_step2.shape[0]}; sum: {ready_data.shape[0] + not_ready_data.shape[0]} vs {stage3_data.shape[0]}")

        # Step 2: Extract based on altLabels
        merge_step2 = not_ready_data[['id', 'classified_title_clean']].merge(esco_exploded, left_on='classified_title_clean', right_on='altLabels', how='left')
        #merge_step2.drop_duplicates(subset=['id'], keep = 'first', inplace=True)

        not_ready_data = not_ready_data.merge(merge_step2[['id', 'preferredLabel']], left_on='id', right_on='id', how='left')
        not_ready_data.rename({'preferredLabel': 'esco_title'}, axis=1, inplace=True)

        step_ready_data = not_ready_data[not_ready_data['esco_title'].notna()].copy()
        step_ready_data['extract_type'] = "altLabels"
        ready_data = pd.concat([ready_data, step_ready_data], ignore_index=True)
        not_ready_data = not_ready_data[not_ready_data['esco_title'].isna()].copy()

        # Step 3: Extract based on preferredLabel with fuzzy search
        not_ready_data['esco_title'] = not_ready_data['classified_title_clean'].apply(
            lambda x: get_best_match(x, esco_labels)
        )
        step_ready_data = not_ready_data[not_ready_data['esco_title'].notna()].copy()
        step_ready_data['extract_type'] = "preferredLabel_fuzzy"
        ready_data = pd.concat([ready_data, step_ready_data], ignore_index=True)
        not_ready_data = not_ready_data[not_ready_data['esco_title'].isna()].copy()

        # Step 4: Extract based on altLabels with fuzzy search
        not_ready_data['esco_title'] = not_ready_data['classified_title_clean'].apply(
            lambda x: get_best_alt_match(x, esco_exploded)
        )

        step_ready_data = not_ready_data[not_ready_data['esco_title'].notna()].copy()
        step_ready_data['extract_type'] = "altLabels_fuzzy"
        ready_data = pd.concat([ready_data, step_ready_data], ignore_index=True)
        not_ready_data = not_ready_data[not_ready_data['esco_title'].isna()].copy()

        # Final step
        ready_data["esco_skills"] = None
        ready_data["esco_id"] = None
        ready_data["esco_code"] = None

        ready_data['esco_id'] = ready_data['esco_title'].apply(
            lambda x: get_esco_id(x, occupations_df)
        )
        ready_data['esco_code'] = ready_data['esco_title'].apply(
            lambda x: get_esco_code(x, occupations_df)
        )

        for _, row in ready_data.iterrows():
            related_skills = occupation_skills_df[occupation_skills_df['occupationUri'] == row["esco_id"]]
            extracted_skills = related_skills.merge(skills_df, left_on='skillUri', right_on='conceptUri')
            ready_data.loc[_, "esco_skills"] = json.dumps(extracted_skills["preferredLabel"].tolist())

        ready_data = pd.concat([ready_data, not_ready_data], ignore_index=True)

        for _, row in ready_data.iterrows():
            skills_mask = skills_df["conceptUri"].isin(row["skill_ids"].split(','))
            skills_matched = skills_df.loc[skills_mask]
            ready_data.loc[_, "skill_labels_en"] = '[' + ', '.join(f"'{s}'" for s in skills_matched["preferredLabel"]) + ']'

        ready_data = ready_data.reset_index(drop=True)
        ready_data.to_pickle(output_path)
        end_time = time.time()
        print(f"Execution time: {end_time - start_time:.6f} seconds - {stage3_data.shape[0]}")
#not_ready_data
#merge_step2
#ready_data

0 ----- Working on ua-2024-01-01 - 100
Execution time: 5.879062 seconds - 100


**Inspection cell** — loads a processed Stage 4 output pickle to verify the enriched column structure. Update the path to `../data/stage_04/processed/output/ua-2024-01-01.pkl` when working with the demo file.

In [16]:
pd.read_pickle("../data/stage_04/processed/output/ua-2024-01-01.pkl")

,id,title,description,min_salary,max_salary,currency,salary_rate,date_created,date_expired,clean_title,...,job_type_score,classified_code,classified_title,skill_labels_en,classified_title_clean,extract_type,esco_title,esco_skills,esco_id,esco_code
0,5569895143508056331,Civil Engineer,Opportunity to work with cutting-edge technolo...,NaN,NaN,USD,hourly,2024-01-01,2024-01-22,civil engineer,...,0.3823,2431,Civil engineer,"['innovation processes', 'create software desi...",civil engineer,preferredLabel,civil engineer,"[""engineering principles"", ""technical drawings...",http://data.europa.eu/esco/occupation/0f5cc4eb...,3112.1
1,5896890515186804550,Cloud Engineer,Exciting opportunity for a skilled professiona...,37000.0,53000.0,USD,hourly,2024-01-01,2024-01-22,cloud engineer,...,0.4736,2136,Cloud engineer,['interact professionally in research and prof...,cloud engineer,preferredLabel,cloud engineer,"[""systems development life-cycle"", ""cloud secu...",http://data.europa.eu/esco/occupation/349ee6f6...,2512.1
2,3462435741587773860,Cloud Engineer,Exciting opportunity for a skilled professiona...,NaN,NaN,None,None,2024-01-01,None,cloud engineer,...,0.4736,2136,Cloud engineer,['interact professionally in research and prof...,cloud engineer,preferredLabel,cloud engineer,"[""systems development life-cycle"", ""cloud secu...",http://data.europa.eu/esco/occupation/349ee6f6...,2512.1
3,5966343075207544576,Cloud Engineer,We are expanding our team and looking for a de...,33000.0,40000.0,UAH,hourly,2024-01-01,2024-01-22,cloud engineer,...,0.4297,2136,Cloud engineer,"['consult with technical staff', 'communicate ...",cloud engineer,preferredLabel,cloud engineer,"[""systems development life-cycle"", ""cloud secu...",http://data.europa.eu/esco/occupation/349ee6f6...,2512.1
4,2056553320732244409,Cloud Engineer,We are expanding our team and looking for a de...,NaN,NaN,USD,hourly,2024-01-01,2024-03-01,cloud engineer,...,0.4297,2136,Cloud engineer,"['consult with technical staff', 'communicate ...",cloud engineer,preferredLabel,cloud engineer,"[""systems development life-cycle"", ""cloud secu...",http://data.europa.eu/esco/occupation/349ee6f6...,2512.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,4013452053751815132,Graphic Designer,We are expanding our team and looking for a de...,NaN,NaN,EUR,monthly,2024-01-01,2024-02-17,graphic designer,...,0.4058,2166,Graphic and multimedia designers,"['consult with technical staff', 'communicate ...",graphic and multimedia designers,None,None,NaN,NaN,NaN
96,7467278593425513802,PR Specialist,Exciting opportunity for a skilled professiona...,NaN,NaN,USD,monthly,2024-01-01,2024-02-15,pr specialist,...,0.5226,2431,Public relations professionals,['interact professionally in research and prof...,public relations professionals,None,None,NaN,NaN,NaN
97,3574473683388644374,Procurement Specialist,We are expanding our team and looking for a de...,42000.0,57000.0,USD,monthly,2024-01-01,2024-01-26,procurement specialist,...,0.4308,2423,Procurement professionals,"['consult with technical staff', 'communicate ...",procurement professionals,None,None,NaN,NaN,NaN
98,3071944447430993509,Recruiter,Opportunity to work with cutting-edge technolo...,NaN,NaN,UAH,monthly,2024-01-01,2024-01-19,recruiter,...,0.3981,2423,Recruiters,"['innovation processes', 'create software desi...",recruiters,None,None,NaN,NaN,NaN


---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.